# 🎙️ Stable Money AI Avatar Agent
**Run this notebook on Google Colab with T4 GPU (free)**

This will:
1. Install all dependencies
2. Download AI models
3. Start your avatar server
4. Give you a public URL via ngrok

⚠️ Make sure Runtime → Change runtime type → T4 GPU is selected before running!

In [ ]:
# ── CELL 1: Check GPU ──
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
!nvidia-smi

In [ ]:
# ── CELL 2: Install dependencies ──
!pip install -q fastapi uvicorn[standard] python-multipart aiofiles
!pip install -q TTS
!pip install -q 'transformers==4.37.2'
!pip install -q 'bangla==0.0.2'
!pip install -q opencv-python-headless
!pip install -q pyngrok
!pip install -q nest_asyncio

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print('✅ Dependencies installed')

In [ ]:
# ── CELL 3: Install Wav2Lip ──
import os

if not os.path.exists('/content/Wav2Lip'):
    !git clone https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip
    !pip install -q -r /content/Wav2Lip/requirements.txt
    # Download Wav2Lip GAN checkpoint
    !pip install -q gdown
    import gdown
    os.makedirs('/content/checkpoints', exist_ok=True)
    gdown.download(
        'https://drive.google.com/uc?id=1j4BgkHOUFNYYFyHMVJ0Dq1Y6Vo5mLyBn',
        '/content/checkpoints/wav2lip_gan.pth', quiet=False
    )
    print('✅ Wav2Lip installed')
else:
    print('✅ Wav2Lip already installed')

In [ ]:
# ── CELL 4: Start Ollama + pull model ──
import subprocess, time, urllib.request, json

# Start Ollama server in background
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# Pull fast model
print('Pulling llama3.2:3b — this takes 2-3 minutes...')
!ollama pull llama3.2:3b
print('✅ Ollama ready')

In [ ]:
# ── CELL 5: Write server.py ──
server_code = '''
import asyncio, base64, io, os, sys, tempfile, time, uuid
from pathlib import Path
from typing import Optional
import cv2
import numpy as np
import torch
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
import soundfile as sf
from TTS.api import TTS

WAV2LIP_AVAILABLE = False
try:
    sys.path.insert(0, "/content/Wav2Lip")
    import audio as wav2lip_audio
    from models import Wav2Lip as Wav2LipModel
    import face_detection
    WAV2LIP_AVAILABLE = True
    print("[server] Wav2Lip available")
except ImportError:
    print("[server] Wav2Lip not available")

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AVATAR_IMG = "/content/avatar.jpg"
WAV2LIP_CKPT = "/content/checkpoints/wav2lip_gan.pth"
VOICE_SAMPLE = "/content/voice_sample.wav"
print(f"[server] Device: {DEVICE}")

class Models:
    tts = None
    wav2lip = None
    face_frames = None
    detector = None

models = Models()

@app.on_event("startup")
async def load_models():
    print("[startup] Loading TTS...")
    _orig = torch.load
    def _patched(*a, **kw):
        kw.setdefault("weights_only", False)
        return _orig(*a, **kw)
    torch.load = _patched

    if Path(VOICE_SAMPLE).exists():
        models.tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(DEVICE)
        print("[startup] XTTS v2 loaded (voice clone active)")
    else:
        models.tts = TTS("tts_models/en/ljspeech/tacotron2-DDC").to(DEVICE)
        print("[startup] Tacotron2 loaded")

    if WAV2LIP_AVAILABLE and Path(WAV2LIP_CKPT).exists():
        ckpt = torch.load(WAV2LIP_CKPT, map_location=DEVICE)
        s = {k.replace("module.",""): v for k,v in ckpt["state_dict"].items()}
        models.wav2lip = Wav2LipModel()
        models.wav2lip.load_state_dict(s)
        models.wav2lip = models.wav2lip.to(DEVICE).eval()
        models.detector = face_detection.FaceAlignment(face_detection.LandmarksType._2D, flip_input=False, device=DEVICE)
        if Path(AVATAR_IMG).exists():
            await preprocess_face(AVATAR_IMG)
        print("[startup] Wav2Lip loaded")

    print("[startup] All models ready")

def synthesize_speech(text):
    buf = io.BytesIO()
    if Path(VOICE_SAMPLE).exists():
        models.tts.tts_to_file(text=text, speaker_wav=VOICE_SAMPLE, language="en", file_path=buf)
    else:
        models.tts.tts_to_file(text=text, file_path=buf)
    buf.seek(0)
    return buf.read()

async def preprocess_face(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (640, 480))
    preds = models.detector.get_detections_for_batch(np.array([img[:,:,::-1]]))
    if not preds or preds[0] is None:
        print("[avatar] No face detected")
        return
    models.face_frames = [img] * 25
    print("[avatar] Face ready")

def generate_lipsync(audio_bytes):
    if not models.wav2lip or not models.face_frames:
        return None
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        f.write(audio_bytes)
        ap = f.name
    op = f"/tmp/{uuid.uuid4().hex}.mp4"
    try:
        wav = wav2lip_audio.load_wav(ap, 16000)
        mel = wav2lip_audio.melspectrogram(wav)
        mel_chunks, idx, fps = [], 0, 25
        mpf = mel.shape[1] / (len(wav)/16000*fps)
        for _ in models.face_frames:
            s=int(idx); mel_chunks.append(mel[:,s:s+16]); idx+=mpf
        frames = (models.face_frames*((len(mel_chunks)//len(models.face_frames))+1))[:len(mel_chunks)]
        gen = []
        for i in range(0,len(mel_chunks),128):
            ib=np.array(frames[i:i+128]); mb=np.array(mel_chunks[i:i+128])
            im=ib.copy(); im[:,ib.shape[1]//2:]=0
            it=torch.FloatTensor(np.transpose(ib,(0,3,1,2))).to(DEVICE)/255.
            imt=torch.FloatTensor(np.transpose(im,(0,3,1,2))).to(DEVICE)/255.
            mt=torch.FloatTensor(np.transpose(mb,(0,3,1,2))).unsqueeze(1).to(DEVICE)
            with torch.no_grad(): p=models.wav2lip(mt,imt)
            p=(p.permute(0,2,3,1).cpu().numpy()*255).astype(np.uint8)
            for pf,orig in zip(p,ib):
                h,w=orig.shape[:2]; pr=cv2.resize(pf,(w,h)); c=orig.copy(); c[h//2:]=pr[h//2:]; gen.append(c)
        h,w=gen[0].shape[:2]
        wr=cv2.VideoWriter(op,cv2.VideoWriter_fourcc(*"mp4v"),fps,(w,h))
        for f in gen: wr.write(f)
        wr.release()
        mx=op.replace(".mp4","_f.mp4")
        os.system(f"ffmpeg -y -i {op} -i {ap} -c:v copy -c:a aac -shortest {mx} -loglevel quiet")
        with open(mx,"rb") as vf: return vf.read()
    finally:
        for p in [ap,op,op.replace(".mp4","_f.mp4")]:
            try: os.remove(p)
            except: pass

@app.websocket("/ws/talk")
async def ws_talk(ws: WebSocket):
    await ws.accept()
    print("[ws] connected")
    try:
        while True:
            data = await ws.receive_json()
            if data.get("type") == "config": continue
            if data.get("type") != "speak": continue
            text = data.get("text","").strip()
            if not text: continue

            prompt = f"""You are Aayush, a warm friendly financial advisor for Stable Money.
Key facts: FDs from 25+ NBFCs with 8-9.5% returns. Sukoon: no-lock-in, 7-8%. Min Rs 1000. DICGC insured up to Rs 5L.
Answer in 2 sentences. Be warm. No links. No bullet points.
Question: {text}"""

            try:
                import urllib.request as ur, json
                req = ur.Request("http://localhost:11434/api/generate",
                    data=json.dumps({"model":"llama3.2:3b","prompt":prompt,"stream":False,
                        "options":{"num_predict":60,"temperature":0.7}}).encode(),
                    headers={"Content-Type":"application/json"})
                with ur.urlopen(req, timeout=30) as r:
                    answer = json.loads(r.read()).get("response","").strip()
            except Exception as e:
                answer = f"Error: {e}"

            await ws.send_json({"type":"text_chunk","text":answer})
            await ws.send_json({"type":"status","msg":"synthesizing"})

            try:
                audio = await asyncio.get_event_loop().run_in_executor(None, synthesize_speech, answer)
                await ws.send_json({"type":"audio","data":base64.b64encode(audio).decode(),"format":"wav"})
                if models.wav2lip and models.face_frames:
                    video = await asyncio.get_event_loop().run_in_executor(None, generate_lipsync, audio)
                    if video:
                        await ws.send_json({"type":"video","data":base64.b64encode(video).decode(),"format":"mp4"})
            except Exception as e:
                await ws.send_json({"type":"error","msg":str(e)})

            await ws.send_json({"type":"done"})
    except WebSocketDisconnect:
        print("[ws] disconnected")

@app.post("/upload/avatar")
async def upload_avatar(file: UploadFile = File(...)):
    with open(AVATAR_IMG,"wb") as f: f.write(await file.read())
    if models.detector: await preprocess_face(AVATAR_IMG)
    return {"status":"ok"}

@app.post("/upload/voice")
async def upload_voice(file: UploadFile = File(...)):
    with open(VOICE_SAMPLE,"wb") as f: f.write(await file.read())
    return {"status":"ok","message":"Voice sample saved"}

@app.get("/health")
def health():
    return {"status":"ok","device":DEVICE,"tts":models.tts is not None,
            "wav2lip":models.wav2lip is not None,"voice_clone":Path(VOICE_SAMPLE).exists()}

if Path("./static").exists():
    app.mount("/", StaticFiles(directory="static",html=True), name="static")
'''

with open('/content/server.py', 'w') as f:
    f.write(server_code)
print('✅ server.py written')

In [ ]:
# ── CELL 6: Clone frontend from GitHub ──
import os
os.makedirs('/content/static', exist_ok=True)

!curl -s https://raw.githubusercontent.com/aayushjha-blip/stable-money-avatar/main/static/index.html -o /content/static/index.html

# Verify
size = os.path.getsize('/content/static/index.html')
print(f'✅ Frontend downloaded ({size} bytes)')

In [ ]:
# ── CELL 7: Upload your avatar photo (run this cell) ──
from google.colab import files
print('Upload your front-facing photo:')
uploaded = files.upload()
for name, data in uploaded.items():
    with open('/content/avatar.jpg', 'wb') as f:
        f.write(data)
    print(f'✅ Avatar saved: {name} ({len(data)} bytes)')

In [ ]:
# ── CELL 8: Upload voice sample (optional but recommended) ──
from google.colab import files
print('Upload your voice_sample.wav (30s recording):')
uploaded = files.upload()
for name, data in uploaded.items():
    with open('/content/voice_sample.wav', 'wb') as f:
        f.write(data)
    print(f'✅ Voice sample saved: {name} ({len(data)} bytes)')

In [ ]:
# ── CELL 9: Set ngrok auth token ──
# Get your free token at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # ← paste your token here

!ngrok config add-authtoken {NGROK_TOKEN}
print('✅ ngrok configured')

In [ ]:
# ── CELL 10: START SERVER + GET PUBLIC URL ──
import nest_asyncio, uvicorn, threading, time
from pyngrok import ngrok
import os

nest_asyncio.apply()
os.chdir('/content')

# Start ngrok tunnel
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url
ws_url = public_url.replace('https://', 'wss://') + '/ws/talk'

print('=' * 60)
print('🌐 PUBLIC URL:', public_url)
print('🔌 WS URL:', ws_url)
print('=' * 60)
print()
print('📱 Share this URL with anyone:', public_url)
print('⚙️  In the app settings, set Server URL to:', ws_url)
print()
print('Starting server...')

# Run uvicorn
config = uvicorn.Config('server:app', host='0.0.0.0', port=8000, log_level='info')
server = uvicorn.Server(config)
await server.serve()